In [1]:
from pathlib import Path

import pandas as pd

In [2]:
paths = sorted(Path("output/input_space").rglob("eval_table.csv"))
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    table.insert(0, "input_space", path.parts[-4])
    tables.append(table)
table = pd.concat(tables, ignore_index=True)
print(table.shape)
table.head()

33
(107, 16)


,input_space,model,repr,clf,dataset,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,flat,flat_mae,patch,attn,aabc_age,3,0.010,0.150,39,"[10.0, 3.0]",train,0.019571,0.993407,0.003822,0.993460,0.003797
1,flat,flat_mae,patch,attn,aabc_age,3,0.010,0.150,39,"[10.0, 3.0]",validation,8.015460,0.471698,0.067432,0.470364,0.068304
2,flat,flat_mae,patch,attn,aabc_age,3,0.010,0.150,39,"[10.0, 3.0]",test,9.620050,0.365385,0.065701,0.370327,0.065185
3,flat,flat_mae,patch,attn,aabc_sex,3,0.003,0.005,29,"[3.0, 0.1]",train,0.007031,1.000000,0.000000,1.000000,0.000000
4,flat,flat_mae,patch,attn,aabc_sex,3,0.003,0.005,29,"[3.0, 0.1]",validation,0.231261,0.931034,0.033168,0.926768,0.035356


In [3]:
table.query("dataset == 'aabc_sex' and input_space == 'schaefer400'")

,input_space,model,repr,clf,dataset,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
71,schaefer400,schaefer400_mae,patch,attn,aabc_sex,3,0.003,0.0500,31,"[3.0, 1.0]",train,0.389087,0.817410,0.018073,0.812660,0.018695
72,schaefer400,schaefer400_mae,patch,attn,aabc_sex,3,0.003,0.0500,31,"[3.0, 1.0]",validation,0.631577,0.724138,0.059122,0.711801,0.061632
73,schaefer400,schaefer400_mae,patch,attn,aabc_sex,3,0.003,0.0500,31,"[3.0, 1.0]",test,0.654087,0.672727,0.063378,0.659091,0.066640
74,schaefer400,schaefer400_mae,patch,linear,aabc_sex,3,0.030,0.0015,42,"[30.0, 0.03]",train,0.489757,0.760085,0.018888,0.752586,0.019659
75,schaefer400,schaefer400_mae,patch,linear,aabc_sex,3,0.030,0.0015,42,"[30.0, 0.03]",validation,0.539825,0.793103,0.051237,0.783851,0.053176
76,schaefer400,schaefer400_mae,patch,linear,aabc_sex,3,0.030,0.0015,42,"[30.0, 0.03]",test,0.622259,0.690909,0.063491,0.687186,0.063933


In [4]:
summary = table.loc[table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["input_space", "repr", "clf"], columns="dataset"
)
summary = summary.loc[(slice(None), "patch", "attn"), :]
summary

acc                                              \
dataset                 aabc_age  aabc_sex hcpya_rest1lr_gender hcpya_task21   
input_space repr  clf                                                          
flat        patch attn  0.365385  0.890909             0.894231     0.988690   
mni_cortex  patch attn  0.603774  0.862069             0.894231     0.969048   
schaefer400 patch attn  0.288462  0.672727             0.730769     0.982143   

                                      acc_std                                 \
dataset                nsd_cococlip  aabc_age  aabc_sex hcpya_rest1lr_gender   
input_space repr  clf                                                          
flat        patch attn     0.303340  0.065701  0.043995             0.029495   
mni_cortex  patch attn     0.275510  0.064581  0.044958             0.030198   
schaefer400 patch attn     0.272542  0.061073  0.063378             0.044547   

                                                  
dataset                hcpya_task21 nsd_cococlip  
input_space repr  clf                             
flat        patch attn     0.001454     0.005370  
mni_cortex  patch attn     0.002525     0.005255  
schaefer400 patch attn     0.001845     0.005303

In [5]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    # return f"{acc*100:.2f} ± {std*100:.2f}"
    return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [6]:
# datasets = ["hcpya_rest1lr_gender", "aabc_sex", "hcpya_task21", "nsd_cococlip"]
datasets = ["aabc_age", "hcpya_rest1lr_gender", "hcpya_task21", "nsd_cococlip"]

repr_, clf = "patch", "attn"

records = []
for model, name in SPACE_NAMES.items():
    record = {"space": name}
    for ds in datasets:
        try:
            acc = summary.loc[(model, repr_, clf), ("acc", ds)]
            std = summary.loc[(model, repr_, clf), ("acc_std", ds)]
            record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
        except KeyError:
            record[DATASET_NAMES.get(ds, ds)] = "-"
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space   | HCP-A Age       | HCP-YA Sex      | HCP-YA Task21   | NSD COCO24      |
|:--------|:----------------|:----------------|:----------------|:----------------|
| parcel  | 28.8 \mypm{6.1} | 73.1 \mypm{4.5} | 98.2 \mypm{0.2} | 27.3 \mypm{0.5} |
| flat    | 36.5 \mypm{6.6} | 89.4 \mypm{2.9} | 98.9 \mypm{0.1} | 30.3 \mypm{0.5} |
| volume  | 60.4 \mypm{6.5} | 89.4 \mypm{3.0} | 96.9 \mypm{0.3} | 27.6 \mypm{0.5} |
\begin{tabular}{lllll}
\toprule
space & HCP-A Age & HCP-YA Sex & HCP-YA Task21 & NSD COCO24 \\
\midrule
parcel & 28.8 \mypm{6.1} & 73.1 \mypm{4.5} & 98.2 \mypm{0.2} & 27.3 \mypm{0.5} \\
flat & 36.5 \mypm{6.6} & 89.4 \mypm{2.9} & 98.9 \mypm{0.1} & 30.3 \mypm{0.5} \\
volume & 60.4 \mypm{6.5} & 89.4 \mypm{3.0} & 96.9 \mypm{0.3} & 27.6 \mypm{0.5} \\
\bottomrule
\end{tabular}

